# 02 - Data Cleaning

## Step 1: Import Libraries & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
np.random.seed(42)

print("✓ Libraries imported successfully!")
print(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Libraries imported successfully!
Execution time: 2026-05-20 11:39:50


## Step 2: Load and Inspect Data

In [2]:
# Load dataset
df = pd.read_csv('ONINE_FOOD_DELIVERY_ANALYSIS.csv')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Dataset loaded successfully!
Shape: 100000 rows × 25 columns

Memory usage: 30.75 MB


In [3]:
# Initial data quality assessment
print("\n" + "="*60)
print("INITIAL DATA QUALITY ASSESSMENT")
print("="*60)

print("\n1. MISSING VALUES BY COLUMN:")
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Pct': (df.isnull().sum().values / len(df) * 100).round(2)
}).sort_values('Missing_Count', ascending=False)
missing_data = missing_data[missing_data['Missing_Count'] > 0]
print(missing_data.to_string(index=False))

print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Columns with missing data: {(df.isnull().sum() > 0).sum()}")


INITIAL DATA QUALITY ASSESSMENT

1. MISSING VALUES BY COLUMN:
             Column  Missing_Count  Missing_Pct
Cancellation_Reason          90969        90.97
       Final_Amount          55697        55.70
       Customer_Age          50093        50.09
        Distance_km          33470        33.47
  Delivery_Time_Min          33359        33.36
        Order_Value          33327        33.33
          Peak_Hour          32962        32.96
    Customer_Gender          24856        24.86
       Payment_Mode          19911        19.91
       Cuisine_Type          16885        16.89
               City          16726        16.73
   Discount_Applied          16715        16.72
               Area          16685        16.68
    Delivery_Rating          16523        16.52
         Order_Time           1998         2.00
         Order_Date           1014         1.01

Total missing values: 461190
Columns with missing data: 16


In [4]:
# Check data types
print("\n2. DATA TYPES:")
print(df.dtypes)

# Check for duplicates
print(f"\n3. DUPLICATE ROWS: {df.duplicated().sum()}")

# Basic statistics for numeric columns
print("\n4. NUMERIC COLUMNS SUMMARY:")
print(df.describe().round(2))


2. DATA TYPES:
Order_ID                   str
Customer_ID                str
Customer_Age           float64
Customer_Gender            str
City                       str
Area                       str
Restaurant_ID              str
Restaurant_Name            str
Cuisine_Type               str
Order_Date                 str
Order_Time                 str
Delivery_Time_Min      float64
Distance_km            float64
Order_Value            float64
Discount_Applied       float64
Final_Amount           float64
Payment_Mode               str
Order_Status               str
Cancellation_Reason        str
Delivery_Partner_ID        str
Delivery_Rating        float64
Restaurant_Rating      float64
Order_Day                  str
Peak_Hour               object
Profit_Margin          float64
dtype: object

3. DUPLICATE ROWS: 0

4. NUMERIC COLUMNS SUMMARY:
       Customer_Age  Delivery_Time_Min  Distance_km  Order_Value  \
count      49907.00           66641.00     66530.00     66673.00   
mean    

## Step 3: Handle Missing Financial Fields

In [5]:
print("\n" + "="*60)
print("HANDLING FINANCIAL FIELDS")
print("="*60)

# Step 3.1: Identify problematic financial rows
print("\nStep 1: Analyzing financial data relationships...")

financial_cols = ['Order_Value', 'Discount_Applied', 'Final_Amount']
all_three_null = df[financial_cols].isnull().all(axis=1).sum()
any_missing_financial = df[financial_cols].isnull().any(axis=1).sum()

print(f"  - Rows with ALL three financial fields null: {all_three_null}")
print(f"  - Rows with ANY financial field null: {any_missing_financial}")

# Step 3.2: Fill using the relationship formula
print("\nStep 2: Filling using Order_Value - Discount_Applied = Final_Amount")

# Fill Final_Amount when Order_Value and Discount exist
mask_fill_final = df['Final_Amount'].isnull() & df['Order_Value'].notnull() & df['Discount_Applied'].notnull()
df.loc[mask_fill_final, 'Final_Amount'] = df.loc[mask_fill_final, 'Order_Value'] - df.loc[mask_fill_final, 'Discount_Applied']
print(f"  - Filled {mask_fill_final.sum()} Final_Amount values from formula")

# Fill Discount_Applied when Order_Value and Final_Amount exist
mask_fill_discount = df['Discount_Applied'].isnull() & df['Order_Value'].notnull() & df['Final_Amount'].notnull()
df.loc[mask_fill_discount, 'Discount_Applied'] = df.loc[mask_fill_discount, 'Order_Value'] - df.loc[mask_fill_discount, 'Final_Amount']
print(f"  - Filled {mask_fill_discount.sum()} Discount_Applied values from formula")

# Fill Order_Value when Discount and Final_Amount exist
mask_fill_order = df['Order_Value'].isnull() & df['Discount_Applied'].notnull() & df['Final_Amount'].notnull()
df.loc[mask_fill_order, 'Order_Value'] = df.loc[mask_fill_order, 'Final_Amount'] + df.loc[mask_fill_order, 'Discount_Applied']
print(f"  - Filled {mask_fill_order.sum()} Order_Value values from formula")


HANDLING FINANCIAL FIELDS

Step 1: Analyzing financial data relationships...
  - Rows with ALL three financial fields null: 5556
  - Rows with ANY financial field null: 55697

Step 2: Filling using Order_Value - Discount_Applied = Final_Amount
  - Filled 11211 Final_Amount values from formula
  - Filled 0 Discount_Applied values from formula
  - Filled 0 Order_Value values from formula


In [6]:
# Step 3.3: Fill remaining financial nulls using median with discount percentage
print("\nStep 3: Filling remaining financial fields using median strategy...")

# Calculate median discount percentage from clean records
clean_financial = df.dropna(subset=financial_cols)
if len(clean_financial) > 0:
    median_discount_pct = (clean_financial['Discount_Applied'] / clean_financial['Order_Value'].clip(lower=0.01)).median()
    median_discount_pct = np.clip(median_discount_pct, 0, 1)  # Ensure between 0-1
    print(f"  - Median discount percentage: {median_discount_pct*100:.2f}%")
else:
    median_discount_pct = 0.05
    print(f"  - No clean financial records, using default discount: {median_discount_pct*100:.2f}%")

median_order_value = df['Order_Value'].median()
median_discount_amt = df['Discount_Applied'].median()

print(f"  - Median order value: {median_order_value:.2f}")
print(f"  - Median discount amount: {median_discount_amt:.2f}")

# Fill Order_Value if still missing
if df['Order_Value'].isnull().sum() > 0:
    df['Order_Value'].fillna(median_order_value, inplace=True)
    print(f"  - Filled {df['Order_Value'].isnull().sum()} remaining Order_Value with median")

# Fill Discount_Applied
if df['Discount_Applied'].isnull().sum() > 0:
    mask_discount = df['Discount_Applied'].isnull()
    df.loc[mask_discount, 'Discount_Applied'] = df.loc[mask_discount, 'Order_Value'] * median_discount_pct
    print(f"  - Filled {mask_discount.sum()} remaining Discount_Applied using median percentage")

# Fill Final_Amount (calculate from Order_Value and Discount)
if df['Final_Amount'].isnull().sum() > 0:
    mask_final = df['Final_Amount'].isnull()
    df.loc[mask_final, 'Final_Amount'] = df.loc[mask_final, 'Order_Value'] - df.loc[mask_final, 'Discount_Applied']
    print(f"  - Calculated {mask_final.sum()} remaining Final_Amount from formula")

print(f"\n  ✓ Financial fields missing values after step 3: {df[financial_cols].isnull().sum().sum()}")


Step 3: Filling remaining financial fields using median strategy...
  - Median discount percentage: 2.91%
  - Median order value: 1197.00
  - Median discount amount: 50.00
  - Filled 33327 remaining Order_Value with median
  - Filled 16715 remaining Discount_Applied using median percentage
  - Calculated 44486 remaining Final_Amount from formula

  ✓ Financial fields missing values after step 3: 72210


C:\Users\smada\AppData\Local\Temp\ipykernel_11872\3160314299.py:22: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Order_Value'].fillna(median_order_value, inplace=True)


## Step 4: Handle Remaining Missing Values

In [7]:
print("\n" + "="*60)
print("HANDLING REMAINING MISSING VALUES - PROFESSIONAL APPROACH")
print("="*60)

# Convert categorical columns to string for filling
categorical_cols = df.select_dtypes(include='category').columns.tolist()
if categorical_cols:
    print(f"\nConverting {len(categorical_cols)} categorical columns to string for filling...")
    for col in categorical_cols:
        df[col] = df[col].astype('string')

# STEP 1: Handle Cancellation_Reason (expected nulls for non-cancelled orders)
print("\nStep 1: CANCELLATION_REASON (special handling)")
cancelled_mask = df['Order_Status'] == 'Cancelled'
non_cancelled_mask = ~cancelled_mask

mask1 = non_cancelled_mask & df['Cancellation_Reason'].isnull()
df.loc[mask1, 'Cancellation_Reason'] = 'None'
print(f"  ✓ Set {mask1.sum():,} non-cancelled orders to 'None'")

mask2 = cancelled_mask & df['Cancellation_Reason'].isnull()
df.loc[mask2, 'Cancellation_Reason'] = 'Unknown'
print(f"  ✓ Set {mask2.sum():,} cancelled order reasons to 'Unknown'")

# STEP 2: Cancelled orders' financial fields
print("\nStep 2: CANCELLED ORDERS FINANCIAL FIELDS")
mask_cancelled_order = cancelled_mask & df['Order_Value'].isnull()
if mask_cancelled_order.sum() > 0:
    df.loc[mask_cancelled_order, 'Order_Value'] = 0
    df.loc[mask_cancelled_order, 'Discount_Applied'] = 0
    df.loc[mask_cancelled_order, 'Final_Amount'] = 0
    print(f"  ✓ Set {mask_cancelled_order.sum():,} cancelled orders' financial fields to 0")

# STEP 3: Professional categorical fill with intelligent imputation
print("\nStep 3: INTELLIGENT CATEGORICAL IMPUTATION")

# 3A. Cuisine_Type: Use modal (most common) cuisine per restaurant
print("\n  3A. Cuisine_Type: Restaurant-level modal cuisine")
if df['Cuisine_Type'].isnull().sum() > 0:
    cuisine_unknown_count = df['Cuisine_Type'].isnull().sum()
    rest_cuisine_map = df[df['Cuisine_Type'] != 'Unknown'].groupby('Restaurant_ID')['Cuisine_Type'].agg(lambda x: x.value_counts().index[0])
    df['Cuisine_Type'] = df.apply(
        lambda row: rest_cuisine_map.get(row['Restaurant_ID'], df['Cuisine_Type'].value_counts().index[0]) 
                    if pd.isna(row['Cuisine_Type']) or row['Cuisine_Type'] == 'Unknown' 
                    else row['Cuisine_Type'], axis=1
    )
    print(f"       ✓ Fixed {cuisine_unknown_count:,} using restaurant modal cuisine")

# 3B. Customer_Gender: Randomly assign based on 50/50 distribution
print("\n  3B. Customer_Gender: Proportional random assignment")
if df['Customer_Gender'].isnull().sum() > 0 or (df['Customer_Gender'].isin(['Unknown', 'Other'])).sum() > 0:
    gender_unknown_count = df['Customer_Gender'].isin(['Unknown', 'Other']).sum()
    mask_gender = df['Customer_Gender'].isin(['Unknown', 'Other', None]) | df['Customer_Gender'].isnull()
    np.random.seed(42)
    df.loc[mask_gender, 'Customer_Gender'] = np.random.choice(['Male', 'Female'], size=mask_gender.sum(), p=[0.5, 0.5])
    print(f"       ✓ Assigned {gender_unknown_count:,} randomly (50% Male, 50% Female)")

# 3C. City: Use modal city from dataset
print("\n  3C. City: Dataset modal city")
if df['City'].isnull().sum() > 0 or (df['City'] == 'Unknown').sum() > 0:
    city_unknown_count = (df['City'].isnull()).sum() + (df['City'] == 'Unknown').sum()
    modal_city = df[df['City'] != 'Unknown']['City'].value_counts().index[0]
    mask_city = (df['City'].isnull()) | (df['City'] == 'Unknown')
    df.loc[mask_city, 'City'] = modal_city
    print(f"       ✓ Fixed {city_unknown_count:,} with modal city: {modal_city}")

# 3D. Area: Use modal area from dataset
print("\n  3D. Area: Dataset modal area")
if df['Area'].isnull().sum() > 0 or (df['Area'] == 'Unknown').sum() > 0:
    area_unknown_count = (df['Area'].isnull()).sum() + (df['Area'] == 'Unknown').sum()
    modal_area = df[df['Area'] != 'Unknown']['Area'].value_counts().index[0]
    mask_area = (df['Area'].isnull()) | (df['Area'] == 'Unknown')
    df.loc[mask_area, 'Area'] = modal_area
    print(f"       ✓ Fixed {area_unknown_count:,} with modal area: {modal_area}")

# 3E. Standard categorical fills
print("\n  3E. Other categorical columns")
categorical_fill = {
    'Payment_Mode': 'UPI',
    'Order_Status': 'Delivered',
}

for col, fill_value in categorical_fill.items():
    if col in df.columns:
        count_before = df[col].isnull().sum()
        if count_before > 0:
            df[col] = df[col].fillna(fill_value)
            print(f"       ✓ {col:20} | Filled {count_before:,} with '{fill_value}'")

# STEP 4: Fill numeric columns with appropriate values
print("\nStep 4: NUMERIC COLUMNS (median/logical fill)")
numeric_fills = {
    'Customer_Age': df['Customer_Age'].median(),
    'Distance_km': df['Distance_km'].median(),
    'Delivery_Time_Min': df['Delivery_Time_Min'].median(),
    'Delivery_Rating': 0,
    'Restaurant_Rating': df['Restaurant_Rating'].median(),
    'Profit_Margin': df['Profit_Margin'].median(),
}

for col, fill_value in numeric_fills.items():
    if col in df.columns:
        count_before = df[col].isnull().sum()
        if count_before > 0:
            df[col] = df[col].fillna(fill_value)
            print(f"  ✓ {col:20} | Filled {count_before:,} with {fill_value:.2f}")

# STEP 5: Fill temporal columns
print("\nStep 5: TEMPORAL COLUMNS")
if 'Order_Date' in df.columns:
    count_date = df['Order_Date'].isnull().sum()
    if count_date > 0:
        date_mode = df['Order_Date'].mode()[0]
        df['Order_Date'] = df['Order_Date'].fillna(date_mode)
        print(f"  ✓ Order_Date | Filled {count_date:,} with '{date_mode}'")

if 'Order_Time' in df.columns:
    count_time = df['Order_Time'].isnull().sum()
    if count_time > 0:
        df['Order_Time'] = df['Order_Time'].fillna('12:00')
        print(f"  ✓ Order_Time | Filled {count_time:,} with '12:00'")

print("\n" + "-"*60)
remaining_nulls = df.isnull().sum().sum()
print(f"TOTAL MISSING VALUES: {remaining_nulls:,}")
if remaining_nulls == 0:
    print("✓✓✓ ALL MISSING VALUES RESOLVED PROFESSIONALLY! ✓✓✓")
else:
    print(f"\n⚠️ Remaining: {remaining_nulls:,}")
    remaining = df.isnull().sum()[df.isnull().sum() > 0]
    print(remaining)



HANDLING REMAINING MISSING VALUES - PROFESSIONAL APPROACH

Step 1: CANCELLATION_REASON (special handling)
  ✓ Set 84,964 non-cancelled orders to 'None'
  ✓ Set 6,005 cancelled order reasons to 'Unknown'

Step 2: CANCELLED ORDERS FINANCIAL FIELDS
  ✓ Set 5,006 cancelled orders' financial fields to 0

Step 3: INTELLIGENT CATEGORICAL IMPUTATION

  3A. Cuisine_Type: Restaurant-level modal cuisine
       ✓ Fixed 16,885 using restaurant modal cuisine

  3B. Customer_Gender: Proportional random assignment
       ✓ Assigned 25,135 randomly (50% Male, 50% Female)

  3C. City: Dataset modal city
       ✓ Fixed 16,726 with modal city: Hyderabad

  3D. Area: Dataset modal area
       ✓ Fixed 16,685 with modal area: South

  3E. Other categorical columns
       ✓ Payment_Mode         | Filled 19,911 with 'UPI'

Step 4: NUMERIC COLUMNS (median/logical fill)
  ✓ Customer_Age         | Filled 50,093 with 39.00
  ✓ Distance_km          | Filled 33,470 with 9.97
  ✓ Delivery_Time_Min    | Filled 33,359

In [8]:

# Handle remaining missing values
print("\nStep 6: REMAINING EDGE CASES")
print("-"*60)

# Fill remaining financial fields for edge cases
remaining_order = df['Order_Value'].isnull().sum()
if remaining_order > 0:
    df['Order_Value'].fillna(df['Order_Value'].median(), inplace=True)
    print(f"  ✓ Order_Value       | Filled {remaining_order:,} remaining")

remaining_discount = df['Discount_Applied'].isnull().sum()
if remaining_discount > 0:
    df['Discount_Applied'].fillna(0, inplace=True)
    print(f"  ✓ Discount_Applied  | Filled {remaining_discount:,} remaining with 0")

remaining_final = df['Final_Amount'].isnull().sum()
if remaining_final > 0:
    df['Final_Amount'] = df.apply(
        lambda row: row['Order_Value'] - row['Discount_Applied'] if pd.isna(row['Final_Amount']) else row['Final_Amount'],
        axis=1
    )
    print(f"  ✓ Final_Amount      | Calculated {remaining_final:,}")

# Fill Peak_Hour properly
remaining_peak = df['Peak_Hour'].isnull().sum()
if remaining_peak > 0:
    df['Peak_Hour'].fillna(False, inplace=True)
    print(f"  ✓ Peak_Hour         | Filled {remaining_peak:,} with False")

# Final check
final_nulls = df.isnull().sum().sum()
if final_nulls == 0:
    print(f"\n✓✓✓ ALL MISSING VALUES COMPLETELY RESOLVED! ✓✓✓")
else:
    print(f"\n⚠️ Still have {final_nulls:,} values")
    print(df.isnull().sum()[df.isnull().sum() > 0])



Step 6: REMAINING EDGE CASES
------------------------------------------------------------
  ✓ Order_Value       | Filled 28,321 remaining
  ✓ Discount_Applied  | Filled 4,660 remaining with 0


C:\Users\smada\AppData\Local\Temp\ipykernel_11872\267732431.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Order_Value'].fillna(df['Order_Value'].median(), inplace=True)
C:\Users\smada\AppData\Local\Temp\ipykernel_11872\267732431.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through c

  ✓ Final_Amount      | Calculated 28,321
  ✓ Peak_Hour         | Filled 32,962 with False

⚠️ Still have 94,264 values
Order_Value         28321
Discount_Applied     4660
Final_Amount        28321
Peak_Hour           32962
dtype: int64


C:\Users\smada\AppData\Local\Temp\ipykernel_11872\267732431.py:27: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Peak_Hour'].fillna(False, inplace=True)


## Step 5: Data Type Conversion

In [9]:
print("FINAL DATA QUALITY VERIFICATION - PEAK_HOUR ANALYSIS")
print("="*60)

# Check Peak_Hour specifically
print("\n1. PEAK_HOUR DISTRIBUTION:")
peak_hour_dist = df['Peak_Hour'].value_counts()
print(peak_hour_dist)
print(f"\n  Peak Hours (TRUE):  {peak_hour_dist.get(True, 0):7,} ({peak_hour_dist.get(True, 0)/len(df)*100:.2f}%)")
print(f"  Off-Peak (FALSE):   {peak_hour_dist.get(False, 0):7,} ({peak_hour_dist.get(False, 0)/len(df)*100:.2f}%)")
print(f"  Missing (NULL):     {df['Peak_Hour'].isnull().sum():7,}")

print("\n2. OVERALL MISSING VALUES CHECK:")
missing_summary = df.isnull().sum()
total_missing = missing_summary.sum()
print(f"  Total missing values: {total_missing}")
if total_missing == 0:
    print(f"  ✓✓✓ DATASET IS 100% COMPLETE!")
else:
    print(f"  Remaining missing values by column:")
    print(missing_summary[missing_summary > 0])

print("\n3. DATA TYPES VERIFICATION:")
print(f"  Peak_Hour dtype: {df['Peak_Hour'].dtype}")
print(f"  Order_Time dtype: {df['Order_Time'].dtype}")
print(f"  Order_Date dtype: {df['Order_Date'].dtype}")


FINAL DATA QUALITY VERIFICATION - PEAK_HOUR ANALYSIS

1. PEAK_HOUR DISTRIBUTION:
Peak_Hour
False    33585
True     33453
Name: count, dtype: int64

  Peak Hours (TRUE):   33,453 (33.45%)
  Off-Peak (FALSE):    33,585 (33.59%)
  Missing (NULL):      32,962

2. OVERALL MISSING VALUES CHECK:
  Total missing values: 94264
  Remaining missing values by column:
Order_Value         28321
Discount_Applied     4660
Final_Amount        28321
Peak_Hour           32962
dtype: int64

3. DATA TYPES VERIFICATION:
  Peak_Hour dtype: object
  Order_Time dtype: str
  Order_Date dtype: str


In [10]:
print("\n" + "="*60)
print("DATA TYPE CONVERSION")
print("="*60)

# IMPORTANT: Use nullable integer types (Int32, Int64) instead of int/int32
# because nullable types can handle NaN values, while native int cannot

type_conversions = {
    'Customer_Age': 'Int32',  # Nullable integer
    'Delivery_Time_Min': 'float32',
    'Distance_km': 'float32',
    'Order_Value': 'float32',
    'Discount_Applied': 'float32',
    'Final_Amount': 'float32',
    'Delivery_Rating': 'Int8',  # Nullable integer
    'Restaurant_Rating': 'float32',
    'Profit_Margin': 'float32',
    'Peak_Hour': 'bool',
    'Customer_Gender': 'category',
    'City': 'category',
    'Area': 'category',
    'Cuisine_Type': 'category',
    'Payment_Mode': 'category',
    'Order_Status': 'category',
    'Cancellation_Reason': 'category'
}

print("\nConverting data types...")
for column, dtype in type_conversions.items():
    if column in df.columns:
        try:
            df[column] = df[column].astype(dtype)
            print(f"  ✓ {column:25} → {dtype}")
        except Exception as e:
            print(f"  ✗ {column:25} → {dtype} (Error: {str(e)[:40]})")

print(f"\nData types after conversion:")
print(df.dtypes)



DATA TYPE CONVERSION

Converting data types...
  ✓ Customer_Age              → Int32
  ✓ Delivery_Time_Min         → float32
  ✓ Distance_km               → float32
  ✓ Order_Value               → float32
  ✓ Discount_Applied          → float32
  ✓ Final_Amount              → float32
  ✓ Delivery_Rating           → Int8
  ✓ Restaurant_Rating         → float32
  ✓ Profit_Margin             → float32
  ✓ Peak_Hour                 → bool
  ✓ Customer_Gender           → category
  ✓ City                      → category
  ✓ Area                      → category
  ✓ Cuisine_Type              → category
  ✓ Payment_Mode              → category
  ✓ Order_Status              → category
  ✓ Cancellation_Reason       → category

Data types after conversion:
Order_ID                    str
Customer_ID                 str
Customer_Age              Int32
Customer_Gender        category
City                   category
Area                   category
Restaurant_ID               str
Restaurant_Name    

## Step 6: Handle Outliers (IQR Method)

In [11]:
print("\n" + "="*60)
print("OUTLIER DETECTION & CAPPING (IQR METHOD)")
print("="*60)

def cap_outliers_iqr(series, column_name, multiplier=1.5):
    """Cap outliers using IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    outlier_count = ((series < lower_bound) | (series > upper_bound)).sum()
    capped_series = series.clip(lower=lower_bound, upper=upper_bound)
    
    return capped_series, outlier_count, lower_bound, upper_bound

# Apply to numeric columns that need it
numeric_cols_to_cap = ['Delivery_Time_Min', 'Distance_km', 'Order_Value', 'Final_Amount']

for col in numeric_cols_to_cap:
    if col in df.columns:
        capped, outlier_count, lower, upper = cap_outliers_iqr(df[col], col)
        df[col] = capped
        print(f"\n  {col}:")
        print(f"    - Outliers capped: {outlier_count}")
        print(f"    - Valid range: [{lower:.2f}, {upper:.2f}]")


OUTLIER DETECTION & CAPPING (IQR METHOD)

  Delivery_Time_Min:
    - Outliers capped: 0
    - Valid range: [-102.50, 325.50]

  Distance_km:
    - Outliers capped: 0
    - Valid range: [-12.29, 41.18]

  Order_Value:
    - Outliers capped: 0
    - Valid range: [-3684.00, 7620.00]

  Final_Amount:
    - Outliers capped: 0
    - Valid range: [-3739.07, 7503.78]


## Step 7: Validate Business Logic & Constraints

In [12]:
print("\n" + "="*60)
print("BUSINESS LOGIC VALIDATION")
print("="*60)

# 7.1: Age validation
print("\n1. AGE VALIDATION (18-80 range):")
invalid_age = ((df['Customer_Age'] < 18) | (df['Customer_Age'] > 80)).sum()
if invalid_age > 0:
    df['Customer_Age'] = df['Customer_Age'].clip(18, 80)
    print(f"   ⚠ Fixed {invalid_age} ages outside valid range")
else:
    print(f"   ✓ All ages valid (18-80 range)")

# 7.2: Rating validation
print("\n2. RATING VALIDATION (1-5 scale):")
invalid_delivery_rating = ((df['Delivery_Rating'] < 1) | (df['Delivery_Rating'] > 5)).sum()
invalid_restaurant_rating = ((df['Restaurant_Rating'] < 1) | (df['Restaurant_Rating'] > 5)).sum()

if invalid_delivery_rating > 0:
    df['Delivery_Rating'] = df['Delivery_Rating'].clip(1, 5)
    print(f"   ⚠ Fixed {invalid_delivery_rating} delivery ratings")
else:
    print(f"   ✓ Delivery ratings valid")

if invalid_restaurant_rating > 0:
    df['Restaurant_Rating'] = df['Restaurant_Rating'].clip(1, 5)
    print(f"   ⚠ Fixed {invalid_restaurant_rating} restaurant ratings")
else:
    print(f"   ✓ Restaurant ratings valid")

# 7.3: Financial logic validation
print("\n3. FINANCIAL LOGIC VALIDATION:")
financial_errors = (df['Final_Amount'] > df['Order_Value']).sum()
negative_final = (df['Final_Amount'] < 0).sum()
negative_discount = (df['Discount_Applied'] < 0).sum()

if financial_errors > 0:
    # Fix: Final_Amount should never exceed Order_Value
    df.loc[df['Final_Amount'] > df['Order_Value'], 'Final_Amount'] = df.loc[df['Final_Amount'] > df['Order_Value'], 'Order_Value']
    print(f"   ⚠ Fixed {financial_errors} cases where Final_Amount > Order_Value")
else:
    print(f"   ✓ Final_Amount ≤ Order_Value (valid)")

if negative_final > 0:
    df.loc[df['Final_Amount'] < 0, 'Final_Amount'] = 0
    print(f"   ⚠ Fixed {negative_final} negative Final_Amount values")
else:
    print(f"   ✓ No negative Final_Amount")

if negative_discount > 0:
    df.loc[df['Discount_Applied'] < 0, 'Discount_Applied'] = 0
    print(f"   ⚠ Fixed {negative_discount} negative Discount_Applied values")
else:
    print(f"   ✓ No negative discounts")

# 7.4: Cancelled order handling
print("\n4. CANCELLED ORDER HANDLING:")
cancelled_count = (df['Order_Status'] == 'Cancelled').sum()
cancelled_with_delivery_rating = ((df['Order_Status'] == 'Cancelled') & (df['Delivery_Rating'] > 0)).sum()

if cancelled_with_delivery_rating > 0:
    df.loc[df['Order_Status'] == 'Cancelled', 'Delivery_Rating'] = 0
    print(f"   ⚠ Set {cancelled_with_delivery_rating} cancelled order ratings to 0")
else:
    print(f"   ✓ Cancelled orders have 0 delivery rating")
print(f"   Total cancelled orders: {cancelled_count}")

# 7.5: Profit margin validation
print("\n5. PROFIT MARGIN VALIDATION (-1 to 1 range):")
invalid_profit = ((df['Profit_Margin'] < -1) | (df['Profit_Margin'] > 1)).sum()
if invalid_profit > 0:
    df['Profit_Margin'] = df['Profit_Margin'].clip(-1, 1)
    print(f"   ⚠ Fixed {invalid_profit} invalid profit margins")
else:
    print(f"   ✓ All profit margins valid (-1 to 1)")


BUSINESS LOGIC VALIDATION

1. AGE VALIDATION (18-80 range):
   ✓ All ages valid (18-80 range)

2. RATING VALIDATION (1-5 scale):
   ⚠ Fixed 16523 delivery ratings
   ⚠ Fixed 18019 restaurant ratings

3. FINANCIAL LOGIC VALIDATION:
   ✓ Final_Amount ≤ Order_Value (valid)
   ⚠ Fixed 773 negative Final_Amount values
   ✓ No negative discounts

4. CANCELLED ORDER HANDLING:
   ⚠ Set 15036 cancelled order ratings to 0
   Total cancelled orders: 15036

5. PROFIT MARGIN VALIDATION (-1 to 1 range):
   ✓ All profit margins valid (-1 to 1)


## Step 8: Standardize Categorical Values

In [13]:
print("\n" + "="*60)
print("INTELLIGENT PEAK_HOUR DERIVATION")
print("="*60)

# APPROACH: Derive Peak_Hour from Order_Time instead of simple fill
# Peak hours in food delivery are typically:
# - Lunch Peak: 11:00 - 14:00 (11 AM - 2 PM)
# - Dinner Peak: 18:00 - 21:00 (6 PM - 9 PM)

print("\nStep 1: Analyzing current Peak_Hour values")
peak_hour_null = df['Peak_Hour'].isnull().sum()
peak_hour_true = (df['Peak_Hour'] == True).sum()
peak_hour_false = (df['Peak_Hour'] == False).sum()

print(f"  Current distribution:")
print(f"    TRUE (peak hours):  {peak_hour_true:7,}")
print(f"    FALSE (off-peak):   {peak_hour_false:7,}")
print(f"    NULL (missing):     {peak_hour_null:7,}")

# Step 2: Extract hour from Order_Time for all orders
print("\nStep 2: Extracting hour from Order_Time")
# Convert Order_Time to hour (assuming format like "12:30")
df['Hour'] = pd.to_numeric(df['Order_Time'].str.split(':').str[0], errors='coerce')

# Calculate distribution of hours
hour_dist = df['Hour'].value_counts().sort_index()
print(f"  Hour distribution (sample):")
print(hour_dist.head(10))

# Step 3: Identify peak hours from existing TRUE values
print("\nStep 3: Identifying peak hours from data")
peak_orders = df[df['Peak_Hour'] == True]
peak_hours_in_data = peak_orders['Hour'].value_counts().sort_index()

if len(peak_hours_in_data) > 0:
    print(f"  Peak hours marked in data: {sorted(peak_hours_in_data.index.tolist())}")
    peak_hour_range = (peak_hours_in_data.index.min(), peak_hours_in_data.index.max())
    print(f"  Range: {int(peak_hour_range[0])}:00 - {int(peak_hour_range[1])}:59")
else:
    # Use industry standard for food delivery
    peak_hour_range = (11, 21)  # 11 AM to 9 PM
    print(f"  No peak hours in data, using industry standard: {peak_hour_range[0]}:00 - {peak_hour_range[1]}:59")

# Step 4: Derive Peak_Hour using domain knowledge
print("\nStep 4: Deriving Peak_Hour from time logic")

# Define peak hours: Lunch (11-14) and Dinner (18-21)
def is_peak_hour(hour):
    if pd.isna(hour):
        return False
    hour = int(hour)
    # Lunch: 11 AM - 2 PM (11-14)
    # Dinner: 6 PM - 9 PM (18-21)
    return (11 <= hour <= 14) or (18 <= hour <= 21)

# Apply to all rows
df['Peak_Hour_Derived'] = df['Hour'].apply(is_peak_hour)

# Compare with existing data
matches = (df['Peak_Hour'] == df['Peak_Hour_Derived']).sum()
mismatches = (df['Peak_Hour'] != df['Peak_Hour_Derived']).sum()

print(f"  Derived vs Existing comparison:")
print(f"    Matches:    {matches:7,}")
print(f"    Mismatches: {mismatches:7,}")

# Step 5: Fill Peak_Hour using the derived logic
print("\nStep 5: Filling Peak_Hour with derived values")
mask_fill = df['Peak_Hour'].isnull()
df.loc[mask_fill, 'Peak_Hour'] = df.loc[mask_fill, 'Peak_Hour_Derived']

print(f"  ✓ Filled {mask_fill.sum():,} null Peak_Hour values using time-based logic")

# Step 6: Verification
print("\nStep 6: Final Peak_Hour distribution")
peak_true_final = (df['Peak_Hour'] == True).sum()
peak_false_final = (df['Peak_Hour'] == False).sum()
peak_null_final = df['Peak_Hour'].isnull().sum()

print(f"  Final distribution:")
print(f"    TRUE (peak hours):  {peak_true_final:7,} ({peak_true_final/len(df)*100:.2f}%)")
print(f"    FALSE (off-peak):   {peak_false_final:7,} ({peak_false_final/len(df)*100:.2f}%)")
print(f"    NULL (remaining):   {peak_null_final:7,}")

if peak_null_final == 0:
    print(f"  ✓✓ All Peak_Hour values successfully derived!")
else:
    # Final fill: any remaining nulls with False
    df['Peak_Hour'].fillna(False, inplace=True)
    print(f"  ✓ Filled remaining {peak_null_final} with False")

# Remove temporary columns
df.drop('Hour', axis=1, inplace=True)
df.drop('Peak_Hour_Derived', axis=1, inplace=True)
print(f"\n  ✓ Cleaned up temporary columns")



INTELLIGENT PEAK_HOUR DERIVATION

Step 1: Analyzing current Peak_Hour values
  Current distribution:
    TRUE (peak hours):   66,415
    FALSE (off-peak):    33,585
    NULL (missing):           0

Step 2: Extracting hour from Order_Time
  Hour distribution (sample):
Hour
0     98002
12     1998
Name: count, dtype: int64

Step 3: Identifying peak hours from data
  Peak hours marked in data: [0, 12]
  Range: 0:00 - 12:59

Step 4: Deriving Peak_Hour from time logic
  Derived vs Existing comparison:
    Matches:     34,261
    Mismatches:  65,739

Step 5: Filling Peak_Hour with derived values
  ✓ Filled 0 null Peak_Hour values using time-based logic

Step 6: Final Peak_Hour distribution
  Final distribution:
    TRUE (peak hours):   66,415 (66.42%)
    FALSE (off-peak):    33,585 (33.59%)
    NULL (remaining):         0
  ✓✓ All Peak_Hour values successfully derived!

  ✓ Cleaned up temporary columns


## Step 9: Final Data Quality Report

In [14]:
print("\n" + "="*60)
print("FINAL DATA QUALITY REPORT")
print("="*60)

# Summary statistics
print("\n1. COMPLETENESS:")
missing_final = df.isnull().sum().sum()
completeness_pct = (1 - missing_final / (len(df) * len(df.columns))) * 100
print(f"   Total missing values: {missing_final}")
print(f"   Data completeness: {completeness_pct:.2f}%")

print("\n2. DUPLICATION:")
duplicates = df.duplicated().sum()
print(f"   Duplicate rows: {duplicates}")

print("\n3. DATASET DIMENSIONS:")
print(f"   Total rows: {len(df):,}")
print(f"   Total columns: {len(df.columns)}")
print(f"   Total data points: {len(df) * len(df.columns):,}")

print("\n4. NUMERIC COLUMNS STATISTICS:")
numeric_stats = df.select_dtypes(include=[np.number]).describe().round(2)
print(numeric_stats)

print("\n5. CATEGORICAL COLUMNS SUMMARY:")
categorical_cols = df.select_dtypes(include=['category', 'object']).columns
for col in categorical_cols:
    print(f"   {col}: {df[col].nunique()} unique values")


FINAL DATA QUALITY REPORT

1. COMPLETENESS:
   Total missing values: 61302
   Data completeness: 97.55%

2. DUPLICATION:
   Duplicate rows: 0

3. DATASET DIMENSIONS:
   Total rows: 100,000
   Total columns: 25
   Total data points: 2,500,000

4. NUMERIC COLUMNS STATISTICS:
       Customer_Age  Delivery_Time_Min  Distance_km  Order_Value  \
count      100000.0          100000.00    100000.00     71679.00   
mean          38.99             124.98        14.28      1936.44   
std            8.74              74.21        10.45      1589.57   
min            18.0              20.00         1.00         0.00   
25%            39.0              58.00         7.76       555.00   
50%            39.0             120.00         9.97      1120.00   
75%            39.0             165.00        21.13      3381.00   
max            60.0             300.00        40.00      5000.00   

       Discount_Applied  Final_Amount  Delivery_Rating  Restaurant_Rating  \
count          95340.00      71679.

C:\Users\smada\AppData\Local\Temp\ipykernel_11872\4090013910.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['category', 'object']).columns


In [15]:
# Check missing values after filling
print("Missing values after cleaning:")
print("\n")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0])

if missing_after.sum() == 0:
    print("\nAll missing values handled successfully!")

Missing values after cleaning:


Order_Value         28321
Discount_Applied     4660
Final_Amount        28321
dtype: int64


## Step 10: Save Cleaned Dataset

In [16]:
# Save cleaned dataset
output_file = 'cleaned_data.csv'
df.to_csv(output_file, index=False)

print("\n" + "="*60)
print("CLEANED DATA SAVED")
print("="*60)
print(f"\n✓ File saved: {output_file}")
print(f"\nDataset Summary:")
print(f"  - Rows: {len(df):,}")
print(f"  - Columns: {len(df.columns)}")
print(f"  - File size: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nReady for Exploratory Data Analysis!")


CLEANED DATA SAVED

✓ File saved: cleaned_data.csv

Dataset Summary:
  - Rows: 100,000
  - Columns: 25
  - File size: 16.09 MB

Ready for Exploratory Data Analysis!


In [17]:
print("\n" + "="*60)
print("MONTH-WISE REVENUE ANALYSIS")
print("="*60)

# Convert Order_Date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

# Extract month and year
df['Year_Month'] = df['Order_Date'].dt.to_period('M')

# Calculate month-wise revenue (sum of Final_Amount for delivered orders)
delivered_orders = df[df['Order_Status'] == 'Delivered'].copy()
monthly_revenue = delivered_orders.groupby('Year_Month')['Final_Amount'].agg(['sum', 'count', 'mean']).reset_index()
monthly_revenue.columns = ['Month', 'Total_Revenue', 'Order_Count', 'Avg_Order_Value']

print("\nMONTHLY REVENUE BREAKDOWN:")
print("-" * 60)
print(f"{'Month':<15} {'Total Revenue':>15} {'Orders':>10} {'Avg Value':>12}")
print("-" * 60)

for idx, row in monthly_revenue.iterrows():
    print(f"{str(row['Month']):<15} ₹{row['Total_Revenue']:>13,.2f} {int(row['Order_Count']):>10} ₹{row['Avg_Order_Value']:>10,.2f}")

print("-" * 60)
print(f"{'TOTAL':<15} ₹{monthly_revenue['Total_Revenue'].sum():>13,.2f} {int(monthly_revenue['Order_Count'].sum()):>10}")
print(f"\nAverage Monthly Revenue: ₹{monthly_revenue['Total_Revenue'].mean():,.2f}")
print(f"Highest Revenue Month: {monthly_revenue.loc[monthly_revenue['Total_Revenue'].idxmax(), 'Month']} (₹{monthly_revenue['Total_Revenue'].max():,.2f})")
print(f"Lowest Revenue Month: {monthly_revenue.loc[monthly_revenue['Total_Revenue'].idxmin(), 'Month']} (₹{monthly_revenue['Total_Revenue'].min():,.2f})")


MONTH-WISE REVENUE ANALYSIS

MONTHLY REVENUE BREAKDOWN:
------------------------------------------------------------
Month             Total Revenue     Orders    Avg Value
------------------------------------------------------------
2024-01         ₹ 9,615,112.00       4807 ₹  2,000.23
2024-02         ₹ 8,930,245.00       4496 ₹  1,986.26
2024-03         ₹ 9,412,906.00       4771 ₹  1,972.94
2024-04         ₹ 9,139,328.00       4631 ₹  1,973.51
2024-05         ₹ 9,538,042.00       4700 ₹  2,029.37
2024-06         ₹ 9,207,320.00       4551 ₹  2,023.14
2024-07         ₹10,921,385.00       5398 ₹  2,023.23
2024-08         ₹ 9,378,328.00       4724 ₹  1,985.25
2024-09         ₹ 9,461,511.00       4708 ₹  2,009.67
2024-10         ₹ 9,157,105.00       4664 ₹  1,963.36
2024-11         ₹ 9,150,914.00       4634 ₹  1,974.73
2024-12         ₹ 9,139,288.00       4559 ₹  2,004.67
------------------------------------------------------------
TOTAL           ₹113,051,480.00      56643

Average Mont